In [1]:
from __future__ import annotations
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Dict, List, Optional, Any, Union, Annotated, Literal
import random
import yaml
from collections import defaultdict, deque
from pathlib import Path
import scoop


In [2]:
from pathlib import Path
from scoop.utils import load_transformation_graph, make_mermaid
from scoop.configuration import transformation_graph

In [3]:
transformation_graph

TransformationGraph(nodes={'person.age': NumericNode(name='person.age', type='numeric', built_distribution=NumericDistribution(name='person.age', mean=70.0, sd=7.0, min_value=50.0, max_value=95.0, as_int=True), active_when=None, mean=70.0, sd=7.0, min_value=50.0, max_value=95.0, as_int=True), 'person.age_group': NumericRangeNode(name='person.age_group', type='numeric_range', built_distribution=None, active_when=None, from_='person.age', ranges=['<60', '60-69', '70-79', '80+']), 'person.year_of_diagnosis': NumericNode(name='person.year_of_diagnosis', type='numeric', built_distribution=NumericDistribution(name='person.year_of_diagnosis', mean=0.0, sd=None, min_value=2015.0, max_value=2020.0, as_int=True), active_when=None, mean=0, sd=None, min_value=2015.0, max_value=2020.0, as_int=True), 'person.years_since_diagnosis': NumericNode(name='person.years_since_diagnosis', type='numeric', built_distribution=NumericDistribution(name='person.years_since_diagnosis', mean=0.0, sd=None, min_value=

In [4]:
# graph = load_transformation_graph(config_path / "probabilistic_transformations.yaml")
print(transformation_graph.order)          # dependency order
print(transformation_graph.nodes)          # correct node types
print(transformation_graph.edges)          # CPT edges
print(transformation_graph.sample_one())   # fully conditional sampling


['person.age', 'person.age_group', 'person.year_of_diagnosis', 'person.years_since_diagnosis', 'person.sex', 'person.smoking_history', 'person.weight', 'condition_occurrence.laterality', 'condition_occurrence.n_stage', 'condition_occurrence.t_8th', 'condition_occurrence.m_stage', 'condition_occurrence.stage_8th', 'condition_episode.group_stage', 'condition_episode.chemotherapy_eligibility', 'condition_episode.key_mutation', 'condition_episode.targetable_mutation', 'treatment_episode.adjuvant_targeted', 'treatment_episode.definitive_sbrt', 'treatment_episode.surgery_port', 'treatment_episode.adjuvant_chemotherapy', 'treatment_episode.neoadjuvant_chemotherapy_w_ici', 'treatment_episode.adjuvant_targeted_w_chemotherapy', 'treatment_episode.chemoradiotherapy', 'treatment_episode.adjuvant_anti_pdl1', 'treatment_episode.adjuvant_anti_pd1', 'treatment_episode.supportive_care_only', 'person.performance_status', 'condition_occurrence.morphology', 'condition_occurrence.cancer_stage', 'condition_

In [5]:
print(make_mermaid(transformation_graph))

flowchart LR

    subgraph condition_occurrence
        condition_occurrence_m_stage["condition_occurrence.m_stage<br/>M0, M1"]:::node_condition_occurrence_m_stage
        classDef node_condition_occurrence_m_stage fill:#C3F9C3,stroke:#333,stroke-width:1px;
        condition_occurrence_n_stage["condition_occurrence.n_stage<br/>N0, N1, N2, N3"]:::node_condition_occurrence_n_stage
        classDef node_condition_occurrence_n_stage fill:#C3F9C3,stroke:#333,stroke-width:1px;
        condition_occurrence_stage_8th["condition_occurrence.stage_8th<br/>9 categories"]:::node_condition_occurrence_stage_8th
        classDef node_condition_occurrence_stage_8th fill:#C3F9C3,stroke:#333,stroke-width:1px;
        condition_occurrence_t_8th["condition_occurrence.t_8th<br/>7 categories"]:::node_condition_occurrence_t_8th
        classDef node_condition_occurrence_t_8th fill:#C3F9C3,stroke:#333,stroke-width:1px;
        condition_occurrence_laterality["condition_occurrence.laterality<br/>left, right"]::

In [6]:
{k: v for k, v in transformation_graph.nodes.items() if getattr(v, "active_when", None)}

{'treatment_episode.adjuvant_targeted': CategoricalNode(name='treatment_episode.adjuvant_targeted', type='categorical', built_distribution=DiscreteDistribution(name=None, probs={'adj_targ': 1.0}), active_when={'condition_episode.resectability': 'operable', 'condition_episode.group_stage': 'early', 'condition_episode.key_mutation': 'km_yes'}, distribution={'adj_targ': 1.0}, labels=['adj_targ']),
 'treatment_episode.definitive_sbrt': CategoricalNode(name='treatment_episode.definitive_sbrt', type='categorical', built_distribution=DiscreteDistribution(name=None, probs={'sbrt': 1.0}), active_when={'condition_episode.group_stage': 'early', 'condition_episode.resectability': 'inoperable'}, distribution={'sbrt': 1.0}, labels=['sbrt']),
 'treatment_episode.surgery_port': CategoricalNode(name='treatment_episode.surgery_port', type='categorical', built_distribution=DiscreteDistribution(name=None, probs={'port': 1.0}), active_when={'condition_episode.resectability': 'operable', 'condition_episode.

In [7]:
activation_edges = []
for name, node in transformation_graph.nodes.items():
    active_when = getattr(node, "active_when", None)
    if not active_when:
        continue

    for upstream in active_when.keys():
        if upstream not in transformation_graph.nodes:
            raise ValueError(
                f"Node '{name}' has active_when dependency on unknown node '{upstream}'"
            )

        activation_edges.append(
            (
                upstream,
                name,
                "active",
                None,
                "activation",   # edge kind
            )
        )

In [8]:
activation_edges

[('condition_episode.resectability',
  'treatment_episode.adjuvant_targeted',
  'active',
  None,
  'activation'),
 ('condition_episode.group_stage',
  'treatment_episode.adjuvant_targeted',
  'active',
  None,
  'activation'),
 ('condition_episode.key_mutation',
  'treatment_episode.adjuvant_targeted',
  'active',
  None,
  'activation'),
 ('condition_episode.group_stage',
  'treatment_episode.definitive_sbrt',
  'active',
  None,
  'activation'),
 ('condition_episode.resectability',
  'treatment_episode.definitive_sbrt',
  'active',
  None,
  'activation'),
 ('condition_episode.resectability',
  'treatment_episode.surgery_port',
  'active',
  None,
  'activation'),
 ('condition_episode.group_stage',
  'treatment_episode.surgery_port',
  'active',
  None,
  'activation'),
 ('condition_episode.key_mutation',
  'treatment_episode.surgery_port',
  'active',
  None,
  'activation'),
 ('condition_episode.resectability',
  'treatment_episode.adjuvant_chemotherapy',
  'active',
  None,
  'ac

In [9]:
active_when

{'condition_episode.targetable_mutation': 'tm_no',
 'person.performance_status_binary': 'v_poor',
 'condition_episode.group_stage': 'metastatic'}